In [6]:
import numpy as np
import json
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve,
    confusion_matrix, classification_report
)


In [7]:
MODEL_PATH = "AC3_Model/model/ac3_model.keras"
model = keras.models.load_model(MODEL_PATH)
print("Model loaded.")


Model loaded.


In [8]:
with open("AC3_Model/preprocessing/preprocessing_config.json") as f:
    pre_cfg = json.load(f)

mean = pre_cfg["mean"]
std = pre_cfg["std"]
window_size = pre_cfg["window_size"]
stride = pre_cfg["stride"]

print("Preprocessing config loaded.")


Preprocessing config loaded.


In [9]:
try:
    with open("AC3_Model/inference/threshold.json") as f:
        thr_cfg = json.load(f)
    threshold = thr_cfg["optimal_threshold"]
    print("Threshold loaded:", threshold)
except:
    threshold = 0.5
    print("Threshold placeholder:", threshold)


Threshold placeholder: 0.5


In [10]:
def preprocess_signal(raw_signal, window_size=100, stride=1):
    """
    raw_signal: array (N, 3)
    return: windows (num_windows, window_size, 3)
    """
    windows = []
    N = len(raw_signal)

    for start in range(0, N - window_size + 1, stride):
        end = start + window_size
        win = raw_signal[start:end]
        windows.append(win)

    return np.array(windows, dtype=np.float32)


def normalize(x, mean, std):
    return (x - mean) / (std + 1e-6)


In [11]:
def ac3_predict(model, signal, mean, std, threshold=0.5):
    # 1. Preprocess
    windows = preprocess_signal(signal)

    # 2. Normalize
    windows = normalize(windows, mean, std)

    # 3. Predict
    preds = model.predict(windows, verbose=0).flatten()

    # 4. Aggregate (max pooling)
    score = float(np.max(preds))

    # 5. Apply threshold
    label = int(score >= threshold)

    return score, label, preds


In [12]:
# Contoh: dataset evaluasi dummy
# Ganti dengan dataset STEAD, GeoNet, K-NET, JMA, atau sinyal real

X_eval = []   # list of signals, each shape (N, 3)
y_eval = []   # list of labels (0/1)

print("Dataset evaluasi siap.")


Dataset evaluasi siap.


In [14]:
import os

root = "/Volumes/Extreme SSD"

for path, dirs, files in os.walk(root):
    for f in files:
        if f.endswith(".npy"):
            print(os.path.join(path, f))


/Volumes/Extreme SSD/stream_stead/output/stead_processed.npy
/Volumes/Extreme SSD/stream_stead/output/._stead_processed.npy


In [23]:
import numpy as np

X_eval = np.load("X_ac2.npy", allow_pickle=True)
y_eval = np.load("y_ac2.npy")

print("X_eval shape:", X_eval.shape)
print("y_eval shape:", y_eval.shape)
print("Label distribusi:", np.bincount(y_eval.astype(int)))


X_eval shape: (2515, 6000, 1)
y_eval shape: (2515,)
Label distribusi: [   0    0    0 2070  413   32]
